# 06 — Percepción (visión) + actuación (software)

**Idea clave:** un pipeline típico de percepción:
1) Adquisición,
2) Preprocesamiento,
3) Extracción de características,
4) Interpretación,
5) Actuación.

Aquí no usamos cámara real para que sea reproducible: creamos una imagen sintética
con un “objeto” y ruido.

Luego detectamos bordes (Sobel) y decidimos si “hay objeto”.


In [ ]:
# ============================================================
# PERCEPCIÓN + ACTUACIÓN (SOFTWARE)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# 1) ADQUISICIÓN: imagen sintética (0..255)
H, W = 120, 160
img = np.ones((H, W), dtype=np.float32) * 30   # fondo

# “Objeto” rectangular (como caja en banda transportadora)
img[40:85, 60:120] = 200

# Ruido (simula mala iluminación / sensor barato)
ruido = np.random.normal(0, 12, (H, W))
img_ruidosa = np.clip(img + ruido, 0, 255)

# 2) PREPROCESAMIENTO: filtro promedio 3x3 (básico, didáctico)
def filtro_promedio_3x3(im):
    salida = np.zeros_like(im)
    for r in range(1, im.shape[0]-1):
        for c in range(1, im.shape[1]-1):
            ventana = im[r-1:r+2, c-1:c+2]
            salida[r, c] = np.mean(ventana)
    return salida

img_suave = filtro_promedio_3x3(img_ruidosa)

# 3) EXTRACCIÓN: Sobel (gradientes)
Kx = np.array([[-1, 0, 1],
               [-2, 0, 2],
               [-1, 0, 1]], dtype=np.float32)

Ky = np.array([[-1,-2,-1],
               [ 0, 0, 0],
               [ 1, 2, 1]], dtype=np.float32)

def conv2d(im, K):
    salida = np.zeros_like(im)
    for r in range(1, im.shape[0]-1):
        for c in range(1, im.shape[1]-1):
            ventana = im[r-1:r+2, c-1:c+2]
            salida[r, c] = np.sum(ventana * K)
    return salida

gx = conv2d(img_suave, Kx)
gy = conv2d(img_suave, Ky)

bordes = np.sqrt(gx**2 + gy**2)

# 4) INTERPRETACIÓN: “hay objeto” si hay suficientes bordes fuertes
umbral = 60
mask_bordes = (bordes > umbral).astype(int)
cantidad_bordes = mask_bordes.sum()

hay_objeto = cantidad_bordes > 300  # regla simple (heurística)

print("Cantidad de píxeles de borde fuerte:", cantidad_bordes)
print("Interpretación: ¿hay objeto?:", hay_objeto)

# 5) ACTUADOR SOFTWARE: registramos evento (simula BD/Kafka/API)
log_eventos = []

def registrar_evento(mensaje):
    log_eventos.append(mensaje)

if hay_objeto:
    registrar_evento("ALERTA: Objeto detectado. Acción: notificar / detener banda.")
else:
    registrar_evento("OK: Sin objeto relevante. Acción: continuar.")

print("\nLOG DEL SISTEMA:")
for e in log_eventos:
    print(" -", e)

# 6) Visualización del pipeline
plt.figure(figsize=(12,3))

plt.subplot(1,4,1)
plt.imshow(img_ruidosa, cmap="gray")
plt.title("Adquisición (ruido)")
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(img_suave, cmap="gray")
plt.title("Preprocesamiento")
plt.axis("off")

plt.subplot(1,4,3)
plt.imshow(bordes, cmap="gray")
plt.title("Bordes (Sobel)")
plt.axis("off")

plt.subplot(1,4,4)
plt.imshow(mask_bordes, cmap="gray")
plt.title("Interpretación (mask)")
plt.axis("off")

plt.show()


## ¿Dónde se usa en industria?
- Inspección visual en líneas de producción.
- Presencia/ausencia de piezas.
- Detección simple de defectos (primer paso antes de modelos más avanzados). 
